# Milestone 3: Retrieval-Augmented Generation (RAG) & Re-ranking

This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without context. Finally, we will explore the physical limits of context windows by tuning the correct number of chunks to retrieve and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

---

## Objectives

- Understand the core concepts and workflow of Retrieval-Augmented Generation (RAG).
- Query a vector index to retrieve relevant context.
- Use a Cross-Encoder to re-rank retrieved results for improved accuracy.
- Compare zero-shot inference with and without retrieved context using A/B testing.
- Explore context-window limitations by tuning the number of chunks retrieved.
- Understand the impact of incorrect or misleading context on LLM outputs.

In [1]:
!pip install faiss-cpu

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

# Load dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base...")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating FAISS index...")
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created!")

# Initialize zero-shot classifier pipeline
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.0 MB/s eta 0:00:00
Creating knowledge base...
Loading embedding model and creating FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
# --- Q1 Code ---
# Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
correct_text_150 = str(row_150[row_150['answer']])

# Predict without context
res_q1 = zs(prompt_150, candidate_labels=labels_150)
correct_score_q1 = res_q1['scores'][res_q1['labels'].index(correct_text_150)]

print(f"Q1 Answer -> Probability score of ground-truth option: {correct_score_q1:.3f}")
# Expected Output: 0.384

Q1 Answer -> Probability score of ground-truth option: 0.384


In [3]:
# --- Q2 Code 
# Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)? 

prompt_150_vec = model.encode([prompt_150])
distances, indices = index.search(prompt_150_vec, k=10)
retrieved_indices_150 = indices[0].tolist()

# Find 1-based rank of KB index 150
rank_q2 = retrieved_indices_150.index(150) + 1

print(f"Q2 Answer -> FAISS rank of true document: {rank_q2}")
# Expected Output: 10

Q2 Answer -> FAISS rank of true document: 10


## Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face)
Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

In [4]:
# --- Q3 Code ---
# Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document? 
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices_150]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Rerank retrieved indices by Cross-Encoder score descending
sorted_pairs = sorted(zip(retrieved_indices_150, ce_scores), key=lambda x: x[1], reverse=True)
reranked_indices = [idx for idx, score in sorted_pairs]

rank_q3 = reranked_indices.index(150) + 1

print(f"Q3 Answer -> Cross-Encoder reranked position: {rank_q3}")
# Expected Output: 1

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Q3 Answer -> Cross-Encoder reranked position: 1


In [5]:
# Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate? *
# --- Q4 Code ---
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
prompt_42_vec = model.encode([prompt_42])

_, indices_42 = index.search(prompt_42_vec, k=5)
retrieved_docs_42 = [kb[i] for i in indices_42[0]]

concatenated_docs = " ".join(retrieved_docs_42)
rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer_bert = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens_q4 = tokenizer_bert(rag_string_42, truncation=False)['input_ids']

print(f"Q4 Answer -> Total token count: {len(tokens_q4)}")
# Expected Output: 216

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 Answer -> Total token count: 216


In [6]:
# Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places). *
# --- Q5 Code ---
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

res_q5 = zs(rag_string_150, candidate_labels=labels_150)
correct_score_q5 = res_q5['scores'][res_q5['labels'].index(correct_text_150)]

print(f"Q5 Answer -> RAG probability of correct option: {correct_score_q5:.3f}")
# Expected Output: 0.989

Q5 Answer -> RAG probability of correct option: 0.989


# Concept: The Danger of Bad Retrieval (Adversarial RAG) 
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

In [7]:
# --- Q6 Code ---
# Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

adv_doc = kb[999]
adv_rag_string = f"Context: {adv_doc} Question: {prompt_150}"

res_q6 = zs(adv_rag_string, candidate_labels=labels_150)
correct_score_q6 = res_q6['scores'][res_q6['labels'].index(correct_text_150)]

print(f"Q6 Answer -> Adversarial RAG probability: {correct_score_q6:.3f}")
# Expected Output: 0.529

Q6 Answer -> Adversarial RAG probability: 0.529


In [8]:
# In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question. 

# Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).
# --- Q7 Code ---
hits = 0
for i in range(100):
    row = train.iloc[i]
    p = str(row['prompt'])
    correct_opt_text = str(row[row['answer']])
    
    p_vec = model.encode([p])
    _, idxs = index.search(p_vec, k=5)
    retrieved_docs = [kb[idx] for idx in idxs[0]]
    
    # Check if correct option text exists within retrieved docs
    if any(correct_opt_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / 100.0) * 100
print(f"Q7 Answer -> Exact Hit Rate Percentage: {hit_rate:.1f}%")
# Expected Output: 73.0%

Q7 Answer -> Exact Hit Rate Percentage: 73.0%


In [9]:
# Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
# For each row, your pipeline must do the following in order:

# Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

# Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

# Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

# Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

# Score:
#  Look at the probability scores output by the model. Rank the options 
# from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

# What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).
# --- Q8 Code ---
def calculate_map3(pred_letters, true_letter):
    for rank, letter in enumerate(pred_letters[:3]):
        if letter == true_letter:
            return 1.0 / (rank + 1)
    return 0.0

options_keys = ['A', 'B', 'C', 'D', 'E']
map3_scores = []

for i in range(20):
    row = train.iloc[i]
    p = str(row['prompt'])
    true_ans_letter = row['answer']
    
    option_texts = [str(row[opt]) for opt in options_keys]
    
    # 1. Retrieve top 5
    p_vec = model.encode([p])
    _, idxs = index.search(p_vec, k=5)
    retrieved_docs = [kb[idx] for idx in idxs[0]]
    
    # 2. Rerank with Cross-Encoder
    pairs = [[p, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc_idx = int(np.argmax(ce_scores))
    best_doc = retrieved_docs[best_doc_idx]
    
    # 3. Augment
    rag_prompt = f"Context: {best_doc} Question: {p}"
    
    # 4. Predict
    zs_res = zs(rag_prompt, candidate_labels=option_texts)
    
    # Map text back to option letters (A-E)
    text_to_letter = {str(row[opt]): opt for opt in options_keys}
    predicted_letters = [text_to_letter[lbl] for lbl in zs_res['labels']]
    
    # 5. Score MAP@3
    score = calculate_map3(predicted_letters, true_ans_letter)
    map3_scores.append(score)

avg_map3 = float(np.mean(map3_scores))
print(f"Q8 Answer -> Average MAP@3 of RAG Pipeline across 20 rows: {avg_map3:.3f}")
# Expected Output: 0.975

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q8 Answer -> Average MAP@3 of RAG Pipeline across 20 rows: 0.975
